Team coursework: SVM classification with class-imbalance handling, feature selection, and hyperparameter tuning. The input CSV and rendered outputs are not included, and results are not independently reproduced in this portfolio copy.


In [ ]:
!pip install optuna

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, precision_recall_curve, average_precision_score
)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.metrics import matthews_corrcoef

In [ ]:
df = df = pd.read_csv("data/datafile_small.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
print(df.isna().sum().sort_values(ascending=False).head())
print(df['target'].value_counts(normalize=True))

df.describe().T.head()

In [ ]:
df = df.drop(columns=['ID_code'])

In [ ]:
df = df.drop(columns=['var_45'])
missing = df.isna().sum().sort_values(ascending=False)
cols_with_missing = missing[missing > 0].index.tolist()

In [ ]:
randomseed = 30
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=randomseed
)

X_train.shape, X_test.shape

In [ ]:
imputer = SimpleImputer(strategy="mean")
imputer.fit(X_train)

X_train_imp = imputer.transform(X_train)
X_test_imp  = imputer.transform(X_test)

print("Imputation done.")

### Balancing

In [ ]:
sm = SMOTE(random_state=randomseed)
X_train_sm, y_train_sm = sm.fit_resample(X_train_imp, y_train)

balancing_sets = {"No Balancing": (X_train_imp, y_train, None),
    "SMOTE": (X_train_sm, y_train_sm, None),
    "Class Weight": (X_train_imp, y_train, "balanced")}


### Evaluate function

In [ ]:
def evaluate_svm(
    X_train, y_train,
    X_test, y_test,
    class_weight=None,
    C=1.0,
    gamma='scale'):
    clf = SVC(
        kernel='rbf',
        probability=True,
        class_weight=class_weight,
        C=C,
        gamma=gamma,
        random_state=randomseed)

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    return {
        "AUC": roc_auc_score(y_test, y_proba),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "Confusion Matrix": confusion_matrix(y_test, y_pred)}

### Baseline / FS / Tuning

In [ ]:
results_baseline = {}
results_fs = {}
results_grid = {}
results_optuna = {}

# Tuning search spaces
grid_params = {
    "C": [0.1, 1, 10, 50, 100],
    "gamma": [0.0001, 0.001, 0.01, 0.1]
}

random_params = {
    "C": np.logspace(-2, 2, 30),
    "gamma": np.logspace(-4, 0, 30)
}

for method, (X_tr_raw, y_tr_raw, cw) in balancing_sets.items():

    print(f"\n=========== {method} ===========")

    # Baseline
    scaler = StandardScaler()
    scaler.fit(X_tr_raw)
    X_tr_scaled = scaler.transform(X_tr_raw)
    X_test_scaled = scaler.transform(X_test_imp)

    results_baseline[method] = evaluate_svm(
            X_tr_scaled, y_tr_raw,
            X_test_scaled, y_test,
            class_weight=cw)

    # Feature selection
    from sklearn.feature_selection import SelectKBest, mutual_info_classif
    selector = SelectKBest(mutual_info_classif, k=50)
    selector.fit(X_tr_raw, y_tr_raw)
    X_tr_fs = selector.transform(X_tr_raw)
    X_test_fs = selector.transform(X_test_imp)

    scaler = StandardScaler()
    scaler.fit(X_tr_fs)
    X_tr_scaled_fs = scaler.transform(X_tr_fs)
    X_test_scaled_fs = scaler.transform(X_test_fs)

    results_fs[method] = evaluate_svm(
            X_tr_scaled_fs, y_tr_raw,
            X_test_scaled_fs, y_test,
            class_weight=cw)

    # Grid Search
    grid = GridSearchCV(
            SVC(kernel="rbf", probability=True, class_weight=cw),
            param_grid=grid_params,
            scoring="roc_auc",
            cv=3,
            n_jobs=-1)
    grid.fit(X_tr_scaled_fs, y_tr_raw)

    bestC = grid.best_params_["C"]
    bestG = grid.best_params_["gamma"]

    results_grid[method] = evaluate_svm(
            X_tr_scaled_fs, y_tr_raw,
            X_test_scaled_fs, y_test,
            class_weight=cw,
            C=bestC,
            gamma=bestG)



    # Optuna
    def objective(trial):
      C = trial.suggest_loguniform("C", 1e-2, 1e2)
      gamma = trial.suggest_loguniform("gamma", 1e-4, 1e0)
      model = SVC(kernel="rbf",probability=True,C=C,gamma=gamma,class_weight=cw)
      scores = cross_val_score(model, X_tr_scaled_fs, y_tr_raw, cv=3, scoring="roc_auc")
      return scores.mean()

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30)

    bestC = study.best_params["C"]
    bestG = study.best_params["gamma"]

    results_optuna[method] = evaluate_svm(
            X_tr_scaled_fs, y_tr_raw,
            X_test_scaled_fs, y_test,
            class_weight=cw,
            C=bestC,
            gamma=bestG)



Baseline

In [ ]:
scaler = StandardScaler()
scaler.fit(X_tr_raw)
X_tr_scaled = scaler.transform(X_tr_raw)
X_test_scaled = scaler.transform(X_test_imp)

results_baseline[method] = evaluate_svm(
        X_tr_scaled, y_tr_raw,
        X_test_scaled, y_test,
        class_weight=cw)

### Feature selction


In [ ]:
selector = SelectKBest(mutual_info_classif, k=50)
selector.fit(X_tr_raw, y_tr_raw)
X_tr_fs = selector.transform(X_tr_raw)
X_test_fs = selector.transform(X_test_imp)

scaler = StandardScaler()
scaler.fit(X_tr_fs)
X_tr_scaled_fs = scaler.transform(X_tr_fs)
X_test_scaled_fs = scaler.transform(X_test_fs)

results_fs[method] = evaluate_svm(
        X_tr_scaled_fs, y_tr_raw,
        X_test_scaled_fs, y_test,
        class_weight=cw)

### Grid Search

In [ ]:
grid = GridSearchCV(
        SVC(kernel="rbf", probability=True, class_weight=cw),
        param_grid=grid_params,
        scoring="roc_auc",
        cv=3,
        n_jobs=-1)
grid.fit(X_tr_scaled_fs, y_tr_raw)

bestC = grid.best_params_["C"]
bestG = grid.best_params_["gamma"]

results_grid[method] = evaluate_svm(
        X_tr_scaled_fs, y_tr_raw,
        X_test_scaled_fs, y_test,
        class_weight=cw,
        C=bestC,
        gamma=bestG)

### Optuna

In [ ]:
def objective(trial):
  C = trial.suggest_loguniform("C", 1e-2, 1e2)
  gamma = trial.suggest_loguniform("gamma", 1e-4, 1e0)
  model = SVC(kernel="rbf",probability=True,C=C,gamma=gamma,class_weight=cw)
  scores = cross_val_score(model, X_tr_scaled_fs, y_tr_raw, cv=3, scoring="roc_auc")
  return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

bestC = study.best_params["C"]
bestG = study.best_params["gamma"]

results_optuna[method] = evaluate_svm(
        X_tr_scaled_fs, y_tr_raw,
        X_test_scaled_fs, y_test,
        class_weight=cw,
        C=bestC,
        gamma=bestG)

In [ ]:
print("Baseline keys =", results_baseline.keys())
print("FS keys =", results_fs.keys())
print("Grid keys =", results_grid.keys())
print("Optuna keys =", results_optuna.keys())

### Final Comparison

In [ ]:
metrics = ["AUC", "Accuracy", "Precision", "Recall", "F1"]
methods = ["No Balancing", "SMOTE", "Class Weight"]

rows = []

for metric in metrics:
    for method in methods:
        rows.append({
            "Metric": metric,
            "Balancing": method,
            "Baseline": results_baseline[method][metric],
            "FS": results_fs[method][metric],
            "Grid": results_grid[method][metric],
            "Optuna": results_optuna[method][metric]
        })

df_compare = pd.DataFrame(rows)
df_compare

As the baseline result, it can clearly show that this is not a suitable model for data prediction in this case.

In [ ]:
methods = ["No Balancing", "SMOTE", "Class Weight"]

rows = []

for method in methods:
    rows.append({
        "Balancing": method,
        "Baseline": results_baseline[method]["MCC"],
        "FS": results_fs[method]["MCC"],
        "Grid": results_grid[method]["MCC"],
        "Optuna": results_optuna[method]["MCC"]
    })

df_compare_MCC = pd.DataFrame(rows)
df_compare_MCC

### Confusion Matrix

In [ ]:
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

all_results = {
    "Baseline": results_baseline,
    "FS": results_fs,
    "Grid": results_grid,
    "Optuna": results_optuna
}

balancing_methods = ["No Balancing", "SMOTE", "Class Weight"]
versions = ["Baseline", "FS", "Grid","Optuna"]


fig, axes = plt.subplots(
    nrows=len(balancing_methods),
    ncols=len(versions),
    figsize=(20, 12))

fig.suptitle("Confusion Matrices for All SVM Models", fontsize=18)

for i, bal in enumerate(balancing_methods):
    for j, ver in enumerate(versions):

        cm = all_results[ver][bal]["Confusion"]

        ax = axes[i, j]
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)

        ax.set_title(f"{bal} - {ver}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()